In [27]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
# Token IDs
I = 1
love = 2
cats = 3

input_sequence = np.array([[I, love, cats]])

print(input_sequence)

[[1 2 3]]


In [3]:
#Create an Embedding layer
embedding = layers.Embedding(
    input_dim=4,
    output_dim=4
)

In [4]:
embedded = embedding(input_sequence)

print("Embedding shape:", embedded.shape)
print(embedded.numpy())

Embedding shape: (1, 3, 4)
[[[ 0.03785313 -0.03710172  0.00934075  0.038234  ]
  [-0.0236662  -0.0383569  -0.02237827  0.0469124 ]
  [ 0.00641609 -0.03604133  0.00999523  0.01384863]]]


In [5]:
#Encoder
encoder = layers.LSTM(
    4,
    return_sequences=True,
    return_state=True
)

In [6]:
encoder_output, encoder_h, encoder_c = encoder(embedded)

print("Encoder output shape:", encoder_output.shape)
print("Encoder hidden state shape:", encoder_h.shape)
print("Encoder cell state shape:", encoder_c.shape)

Encoder output shape: (1, 3, 4)
Encoder hidden state shape: (1, 4)
Encoder cell state shape: (1, 4)


In [7]:
#Decoder
decoder_input = np.array([[0]]) #Here 0 can represent a special <START> token.

In [8]:
decoder_embedding = layers.Embedding(
    input_dim=4,
    output_dim=4
)

In [9]:
decoder_embedded = decoder_embedding(decoder_input)

In [10]:
decoder = layers.LSTM(
    4,
    return_sequences=True,
    return_state=True
)

In [11]:
decoder_output, decoder_h, decoder_c = decoder(
    decoder_embedded,
    initial_state=[encoder_h, encoder_c]
)

In [12]:
print("Decoder output:", decoder_output.shape)
print("Decoder hidden state:", decoder_h.shape)

Decoder output: (1, 1, 4)
Decoder hidden state: (1, 4)


In [13]:
#NOW we reach the attention part

#score
scores = tf.matmul(
    decoder_h[:, tf.newaxis, :],
    encoder_output,
    transpose_b=True
)

In [14]:
print("Alignment scores:")
print(scores.numpy())

Alignment scores:
[[[2.7528102e-05 2.0469932e-04 1.9412080e-04]]]


In [15]:
#Convert scores to attention weights
attention_weights = tf.nn.softmax(scores, axis=-1)

print("Attention weights:")
print(attention_weights.numpy())

Attention weights:
[[[0.33329514 0.33335418 0.33335066]]]


In [16]:
#Calculate the context vector
context_vector = tf.matmul(
    attention_weights,
    encoder_output
)

In [17]:
context_vector

<tf.Tensor: shape=(1, 1, 4), dtype=float32, numpy=
array([[[ 0.0064178 , -0.00904954,  0.00949567, -0.00388285]]],
      dtype=float32)>

# Practical 2

In [18]:
I=1
love=2
cats=3

inputs = np.array([[1,2,3]])

In [19]:
#embedding
embedding = layers.Embedding(input_dim=4,output_dim=4)
embedded = embedding(inputs)

In [20]:
encoder = layers.LSTM(4,return_sequences=True,return_state=True)

In [21]:
encoder_output,encoder_h,encoder_c = encoder(embedded)

In [22]:
#decoder
decoder_input = np.array([[0]])
decoder_embedding = layers.Embedding(input_dim=4,output_dim=4)
decoder_embedded = decoder_embedding(decoder_input)

In [23]:
decoder = layers.LSTM(4,return_sequences=True,return_state=True)

In [25]:
decoder_output,decoder_h,decoder_c = decoder(decoder_embedded, initial_state=[encoder_c,encoder_h])

In [ ]:
#score
scores = tf.matmul(decoder_h[:,tf.newaxis,:], encoder_output, transpose_b=True)

In [29]:
attention = tf.nn.softmax(scores,axis=-1)

In [30]:
context_vector = tf.matmul(attention,encoder_output)